# 📊 Financial Report Summarizer - Demo Notebook

**Problem:** Financial reports are long (2+ pages) and full of jargon (EBITDA, impairment, bps...). A common person cannot quickly understand them.

**Solution:** Paste the report → get:
1. **Simple bullet-point summary** (BART - pre-trained Transformer)
2. **Overall verdict: good / bad / mixed news** + key good news & concerns (**BiLSTM** - trained by us)
3. **Difficult words explained** in plain English (jargon dictionary)

```
report text ──► clean & split ──┬──► BART summarizer ──► jargon simplifier ──► bullet points
                                ├──► BiLSTM (per sentence) ──► verdict, good news, concerns
                                └──► jargon finder ──► glossary
```

▶ **Run the cells from top to bottom** (Shift + Enter).

## Step 1 - Setup
Make the `src/` folder importable from this notebook.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src import config
print("Project folder:", PROJECT_ROOT)
print("BiLSTM already trained?", config.BILSTM_WEIGHTS_PATH.exists())

## Step 2 - Train the BiLSTM (only needed once)
Trains on **Financial PhraseBank** (~4,800 financial news sentences labelled positive / neutral / negative by experts).
Takes ~2-5 minutes on a CPU. **Skip this cell if the model is already trained** (Step 1 printed `True`).

In [ ]:
TRAIN_AGAIN = False   # set to True to re-train even if a model already exists

if TRAIN_AGAIN or not config.BILSTM_WEIGHTS_PATH.exists():
    sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
    import train_bilstm
    train_bilstm.main()
else:
    print("Model already trained - skipping. Set TRAIN_AGAIN = True to re-train.")

## Step 3 - Look inside the BiLSTM
The architecture of our own model:

In [ ]:
from src.sentiment import SentimentAnalyzer

analyzer = SentimentAnalyzer()
print(analyzer.model)
print("\nTrainable parameters:", sum(p.numel() for p in analyzer.model.parameters()))

### Quick test of the BiLSTM on single sentences

In [ ]:
test_sentences = [
    "Net profit rose 25% compared to last year, beating analyst expectations.",
    "The company reported a net loss and suspended its dividend.",
    "The annual general meeting will be held in Helsinki on 15 March.",
    "Operating profit fell to EUR 3.1 mn from EUR 5.4 mn.",
    "The company expects profit to grow next year.",
]
for p in analyzer.predict(test_sentences):
    print(f"{p['label']:<9} ({p['confidence']:.0%})  {p['sentence']}")

## Step 4 - Load the full pipeline (BiLSTM + BART)
The **first time**, BART (~1.2 GB) is downloaded from HuggingFace. After that it loads from cache in a few seconds.

In [ ]:
from src.pipeline import FinancialReportPipeline, print_result

pipe = FinancialReportPipeline()
print("Pipeline ready ✔")

## Step 5 - ✍️ INPUT: paste your report here
Paste any financial report text between the triple quotes `""" ... """`.
If you leave it empty, a sample report from `data/sample_reports/` is used.

In [ ]:
REPORT_TEXT = """

"""

# ---- fallback: use a sample report if nothing was pasted ----
if not REPORT_TEXT.strip():
    sample_file = config.SAMPLE_REPORTS_DIR / "01_good_quarter_tech_company.txt"
    # try also: "02_bad_year_retail_company.txt"
    REPORT_TEXT = sample_file.read_text(encoding="utf-8")
    print("Using sample report:", sample_file.name)

print(len(REPORT_TEXT.split()), "words in the input report")

## Step 6 - ✅ OUTPUT: the simple summary

In [ ]:
result = pipe.run(REPORT_TEXT)
print_result(result)

## Step 7 - (Optional) Visualise the sentiment of every sentence

In [ ]:
import matplotlib.pyplot as plt

counts = result["counts"]
colors = {"positive": "#2e9e5b", "neutral": "#9aa0a6", "negative": "#d64545"}
plt.figure(figsize=(6, 3))
plt.bar(counts.keys(), counts.values(), color=[colors[k] for k in counts])
plt.title("How many sentences are good / neutral / bad news (BiLSTM)")
plt.ylabel("number of sentences")
plt.show()

## Summary of the approach
| Component | Type | Trained by us? | Role |
|---|---|---|---|
| **BiLSTM** | Recurrent neural network (PyTorch) | ✅ Yes - Financial PhraseBank | Sentence sentiment → verdict, good news, concerns |
| **DistilBART** | Pre-trained Transformer (encoder-decoder) | ❌ No - pre-trained on CNN/DailyMail | Abstractive summary |
| **Jargon dictionary** | Rule-based | - | Explains financial terms in plain words |

👉 For a nicer interface run: `streamlit run app/app.py`